In [2]:
# SETUPENV
import json

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from config import config
from pathlib import Path

### Check Search Index

In [4]:
sc = SearchClient(config.ai_search.endpoint, 
                  config.ai_search.index_name, 
                #   "rag-index",
                  AzureKeyCredential(config.ai_search.api_key))
print("doc count:", sc.get_document_count())

ResourceNotFoundError: () The index 'rag-index-24-11' for service 'cognitivessvc-clchatbot-search-dev' was not found.
Code: 
Message: The index 'rag-index-24-11' for service 'cognitivessvc-clchatbot-search-dev' was not found.

### Test hybrid query and check on output

In [5]:
%%capture cap

# results = sc.search(search_text="Can I have the template for Simplified SA for Non-Core Consultants?", top=5, select="simplified")
results = sc.search(
    search_text="Simplified SA",
    top=5,
    search_fields=["filepath"],         
    select=["title", "content", "page", "filepath"],
)
for r in results:
    print("=" * 80)
    print(
        r["title"], "\n",
        r["content"],
        # r["page"], 
        # r["filepath"],
        # r["@search.score"], r["content"]
    )

with open("output.txt", 'w') as filee:
    filee.write(cap.stdout)

### Validate Queries against retrieved file

In [5]:
%%capture captured_output

query_list_path = Path('../data/validation/query-list.json')
queries = json.loads(query_list_path.read_text(encoding='utf-8'))
print(f"Loaded {len(queries)} queries from {query_list_path}")

for idx, q in enumerate(queries, start=1):
    sn = q.get('sn', idx)
    query_text = q.get('query', '')
    expected = q.get('file') or q.get('source_file') or q.get('file_expected')

    print()
    print('='*80)
    print(f"[{idx}] Query: {query_text}")

    top5 = list(sc.search(search_text=query_text, top=10))

    if not top5:
        print("No results returned.")
        continue

    print('\nTop 5 retrieved titles:')
    for i, r in enumerate(top5, start=1):
        if isinstance(r, dict):
            print("we are here")
            title = r.get('title') or '<no title>'
            content = r.get('content')
            filepath = r.get('filepath') or r.get('Url') or r.get('url') or '<no filepath>'
        else:
            title = getattr(r, 'title', '<no title>')
            filepath = getattr(r, 'filepath', None) or getattr(r, 'Url', None) or '<no filepath>'
        print(f"{i}. {title}, \n {content} \n filepath: ({filepath})")
        break

    print(f"\nExpected file: {expected}")

with open('output.txt', 'w') as file:
    file.write(captured_output.stdout)


ResourceNotFoundError: () The index 'rag-index-24-11' for service 'cognitivessvc-clchatbot-search-dev' was not found.
Code: 
Message: The index 'rag-index-24-11' for service 'cognitivessvc-clchatbot-search-dev' was not found.

### Delete entire search index

In [ ]:
sic = SearchIndexClient(config.ai_search.endpoint, AzureKeyCredential(config.ai_search.api_key))

sic.delete_index(config.ai_search.index_name)
print(f"Index '{config.ai_search.index_name}' deleted")

### Document Intelligence Testing

In [ ]:
dic = DocumentIntelligenceClient(config.doc_intelligence.endpoint, AzureKeyCredential(config.doc_intelligence.api_key))
file_path = Path("../data/00 Chatbot on PDDM Info - Query List.pdf")

with open(file_path, "rb") as f:
    poller = dic.begin_analyze_document(
        model_id="prebuilt-layout",
        body=f,
        output_content_format=DocumentContentFormat.MARKDOWN,
        content_type="application/pdf",
    )
result = poller.result()

print(result.tables)

In [ ]:
print(result)